# train_discretion_zlb
Standalone training notebook for `discretion_zlb` policy.

In [ ]:
import os, sys, json
import numpy as np
import torch
import pathlib

def _find_project_root():
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir():
        return cand
    raise RuntimeError("Could not find project root containing src/. If on Colab, clone repo to /content/econml.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelParams, TrainConfig
from src.deqn import PolicyNetwork, Trainer, simulate_paths
from src.io_utils import make_run_dir, save_run_metadata, save_selected_run, pack_config, save_torch, save_csv, save_json, save_npz
from src.metrics import residual_quality
from src.sss_from_policy import switching_policy_sss_by_regime_from_policy
from src.sanity_checks import fixed_point_check, residuals_check_switching_consistent

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
params = ModelParams(device=DEVICE, dtype=torch.float32)

cfg_seed = 151082
cfg_probe = TrainConfig.author_like(policy="discretion_zlb", seed=cfg_seed)
run_dir = make_run_dir(ARTIFACTS_ROOT, "discretion_zlb", tag=cfg_probe.mode, seed=cfg_probe.seed)
cfg = TrainConfig.author_like(policy="discretion_zlb", seed=cfg_seed, run_dir=run_dir, artifacts_root=ARTIFACTS_ROOT)

rbar_by_regime = None

save_run_metadata(run_dir, pack_config(params, cfg, extra={"policy":"discretion_zlb"}))
print("Run dir:", run_dir)

In [ ]:
# ---------- model ----------
d_in, d_out = 5, 6
net = PolicyNetwork(d_in, d_out, hidden=cfg.hidden_layers, activation=cfg.activation, init_mode=cfg.init_mode, init_scale=cfg.init_scale, seed=cfg.seed)

trainer = Trainer(
    params=params,
    cfg=cfg,
    policy="discretion_zlb",
    net=net,
    rbar_by_regime=rbar_by_regime,
)

In [ ]:
# ---------- train ----------
losses = trainer.train(
    commitment_sss=None,
    n_path=cfg.n_path,
    n_paths_per_step=cfg.n_paths_per_step,
)

save_torch(os.path.join(run_dir, "weights.pt"), trainer.net.state_dict())
import pandas as pd
df = pd.DataFrame({"iter": np.arange(len(losses)), "loss": losses})
save_csv(os.path.join(run_dir, "train_log.csv"), df)

ctx = torch.enable_grad() if trainer.policy in ("discretion", "discretion_zlb") else torch.inference_mode()
with ctx:
    x_val = trainer.simulate_initial_state(int(cfg.val_size), commitment_sss=None)
    val_burn = int(getattr(cfg, "val_burn_in", 200))
    for _ in range(val_burn):
        x_val = trainer._step_state(x_val)
    resid = trainer._residuals(x_val).detach().cpu().numpy()
q = residual_quality(resid, tol=getattr(cfg, "report_tol", 1e-3))
save_json(os.path.join(run_dir, "train_quality.json"), q)
print("Train quality:", q)

sss_pol = switching_policy_sss_by_regime_from_policy(
    params,
    trainer.net,
    policy="discretion_zlb",
    rbar_by_regime=rbar_by_regime,
)
save_json(os.path.join(run_dir, "sss_policy_fixed_point.json"), {"policy": "discretion_zlb", "by_regime": sss_pol.by_regime})

fp = fixed_point_check(
    params,
    trainer.net,
    policy="discretion_zlb",
    sss_by_regime=sss_pol.by_regime,
    rbar_by_regime=rbar_by_regime,
)
rc = residuals_check_switching_consistent(
    params,
    trainer.net,
    policy="discretion_zlb",
    sss_by_regime=sss_pol.by_regime,
    rbar_by_regime=rbar_by_regime,
)

save_json(os.path.join(run_dir, "sanity_checks.json"), {
    "policy": "discretion_zlb",
    "fixed_point_max_abs_state_diff": {int(k): float(v.max_abs_state_diff) for k,v in fp.items()},
    "residual_max_abs": {int(k): float(v.max_abs_residual) for k,v in rc.items()},
    "residuals_by_regime": {int(k): {kk: float(vv) for kk,vv in v.residuals.items()} for k,v in rc.items()},
})

# ---------- simulate ergodic paths ----------
x0 = trainer.simulate_initial_state(512, commitment_sss=None)
sim = simulate_paths(
    params=params,
    policy=trainer.policy,
    net=trainer.net,
    T=20000,
    burn_in=2000,
    x0=x0,
    rbar_by_regime=rbar_by_regime,
    compute_implied_i=True,
    gh_n=3,
    thin=1,
    store_states=True,
    show_progress=True,
)
save_npz(os.path.join(run_dir, "sim_paths.npz"), **sim)
print("Saved sim_paths to:", os.path.join(run_dir, "sim_paths.npz"))

save_selected_run(ARTIFACTS_ROOT, trainer.policy, run_dir)
print("Selected run saved for discretion_zlb:", run_dir)


In [ ]:
# ---------- Long-sim moments (our diagnostic object) ----------
from src.moment_reports import print_sim_paths_moments_report
sim_report = print_sim_paths_moments_report(
    run_dir=run_dir,
    device=DEVICE,
    dtype=params.dtype,
)


In [ ]:
# ---------- Author NT/SS moments (table-comparison object) ----------
from src.moment_reports import build_and_print_author_nt_ss_report
author_report = build_and_print_author_nt_ss_report(
    run_dir=run_dir,
    policy=trainer.policy,
    artifacts_root=ARTIFACTS_ROOT,
    device=DEVICE,
    dtype=params.dtype,
    cons_mode="paper",
    rebuild_author_postprocess=False,
)
